This notebook is self-contained except for importing `model_profiler.py`.  
The `Constant` accounting bug is already documented elsewhere, so it is not expanded here.

In [ ]:
!pip install -q numpy==2.4.4 2>/dev/null
!pip install -q onnx==1.21.0 2>/dev/null
!pip install -q onnxruntime==1.24.4 2>/dev/null
!pip install -q onnx-tool==1.0.1 2>/dev/null

In [ ]:
import json
import math
import sys
from pathlib import Path
from zipfile import ZipFile

import numpy as np
import onnx
import onnxruntime
from IPython.display import display
from onnx import TensorProto, helper, numpy_helper

WORK_DIR = Path("/kaggle/working/")
DATA_DIR = Path("/kaggle/input/competitions/neurogolf-2026/")
sys.path.insert(0, '/kaggle/input/datasets/pavelsavchenkov/model-profiler-apr-25')
from model_profiler import compute_cost, profile_frame, profile_model, runtime_details, score_network_details

MODEL_SHAPE = (1, 10, 30, 30)

In [ ]:
def tensor(name, values):
    return numpy_helper.from_array(np.asarray(values), name=name)


def constant_node(name, output, values):
    return helper.make_node("Constant", [], [output], name=name, value=tensor(name, values))


def save_model(model, filename):
    path = WORK_DIR / filename
    onnx.save(model, path)
    return path


def score_from_cost(cost):
    return max(1.0, 25.0 - math.log(float(cost)))


def load_task(task_num):
    path = DATA_DIR / f"task{task_num:03d}.json"
    return json.loads(path.read_text())


def grid_to_tensor(grid):
    board = np.asarray(grid, dtype=np.int64)
    height, width = board.shape
    tensor_value = np.zeros(MODEL_SHAPE, dtype=np.float32)
    for row in range(height):
        for col in range(width):
            tensor_value[0, board[row, col], row, col] = 1.0
    return tensor_value


def tensor_to_grid(output_tensor, height, width):
    thresholded = (np.asarray(output_tensor) > 0).astype(np.float32)
    return np.argmax(thresholded[0, :, :height, :width], axis=0).astype(int).tolist()


def verify_examples(task_num, model_path):
    task = load_task(task_num)
    session = onnxruntime.InferenceSession(str(model_path), providers=["CPUExecutionProvider"])
    total_examples = 0
    passed_examples = 0
    for split_name in ("train", "test", "arc-gen"):
        for example in task.get(split_name, []):
            expected = example["output"]
            output_tensor = session.run(["output"], {"input": grid_to_tensor(example["input"])})[0]
            predicted = tensor_to_grid(output_tensor, len(expected), len(expected[0]))
            total_examples += 1
            passed_examples += int(predicted == expected)
    return passed_examples, total_examples


def write_submission(model_path, task_num, label):
    submission_dir = WORK_DIR / f"{label}_submission"
    submission_dir.mkdir(parents=True, exist_ok=True)
    model_target = submission_dir / f"task{task_num:03d}.onnx"
    model_target.write_bytes(Path(model_path).read_bytes())
    zip_path = submission_dir / "submission.zip"
    if zip_path.exists():
        zip_path.unlink()
    with ZipFile(zip_path, "w") as zip_file:
        zip_file.write(model_target, arcname=model_target.name)
    print({"submission_dir": str(submission_dir), "submission_zip": str(zip_path)})
    return submission_dir, zip_path


def show_profile(model_path):
    totals, _ = profile_model(model_path)
    cost = int(totals["macs"] + totals["memory"] + totals["params"])
    print(f"Profile totals: macs={totals['macs']} memory={totals['memory']} params={totals['params']} cost={cost}")
    display(profile_frame(model_path))


def show_expected_vs_actual(expected_text, actual_text):
    print("Expected in a correct profile:")
    print(expected_text)
    print("Actual in onnx-tool:")
    print(actual_text)


def report_real_task(task_num, model_path):
    passed_examples, total_examples = verify_examples(task_num, model_path)
    cost = compute_cost(model_path)
    print(f"Task {task_num}")
    print(f"Passed examples: {passed_examples}/{total_examples}")
    print(f"Cost: {int(cost)}")
    print(f"Score: {score_from_cost(cost):.15f}")


def show_demo(model_path, sample_input=None):
    details = score_network_details(model_path)
    runtime = runtime_details(model_path, sample_input)
    print("score_network_details:")
    print(details)
    print("runtime_details:")
    print(runtime)


PLACEMENTS = {
    0: ((0, 0), (4, 4)),
    1: ((0, 5), (4, 1)),
    2: ((1, 4), (5, 0)),
    3: ((1, 1), (5, 5)),
}


def output_bank():
    bank = np.zeros((4, 10, 30, 30), dtype=np.bool_)
    for quadrant, starts in PLACEMENTS.items():
        bank[quadrant, 0, :9, :9] = True
        for row0, col0 in starts:
            bank[quadrant, 0, row0 : row0 + 4, col0 : col0 + 4] = False
            bank[quadrant, 3, row0 : row0 + 4, col0 : col0 + 4] = True
    return bank


def make_model(nodes, name, initializers=None):
    graph = helper.make_graph(
        nodes,
        name,
        [helper.make_tensor_value_info("input", TensorProto.FLOAT, MODEL_SHAPE)],
        [helper.make_tensor_value_info("output", TensorProto.BOOL, MODEL_SHAPE)],
        initializer=[] if initializers is None else initializers,
    )
    model = helper.make_model(
        graph,
        producer_name=name,
        opset_imports=[helper.make_opsetid("", 16)],
    )
    onnx.checker.check_model(model)
    return model


def green_crop_nodes():
    return [
        constant_node("green_crop_starts", "green_crop_starts", np.array([3, 0, 0], dtype=np.int64)),
        constant_node("green_crop_ends", "green_crop_ends", np.array([4, 3, 3], dtype=np.int64)),
        constant_node("green_crop_axes", "green_crop_axes", np.array([1, 2, 3], dtype=np.int64)),
        constant_node("green_crop_steps", "green_crop_steps", np.array([1, 1, 1], dtype=np.int64)),
        helper.make_node(
            "Slice",
            ["input", "green_crop_starts", "green_crop_ends", "green_crop_axes", "green_crop_steps"],
            ["green_3x3"],
            name="green_3x3",
        ),
    ]


def reshape_bool_compress_tail(condition_source):
    return [
        constant_node("reshape4", "reshape4", np.array([4], dtype=np.int64)),
        helper.make_node("Reshape", [condition_source, "reshape4"], ["corners_flat"], name="corners_flat"),
        helper.make_node("Cast", ["corners_flat"], ["condition"], name="condition", to=TensorProto.BOOL),
        helper.make_node("Compress", ["bank", "condition"], ["output"], name="select_bank", axis=0),
    ]


def build_task104_resize():
    nodes = [
        constant_node("bank_value", "bank", output_bank()),
        *green_crop_nodes(),
        constant_node("resize_sizes", "resize_sizes", np.array([1, 1, 2, 2], dtype=np.int64)),
        helper.make_node(
            "Resize",
            ["green_3x3", "", "", "resize_sizes"],
            ["corners_2x2"],
            name="resize_to_corners",
            mode="nearest",
            coordinate_transformation_mode="align_corners",
            nearest_mode="floor",
        ),
        *reshape_bool_compress_tail("corners_2x2"),
    ]
    return make_model(nodes, "task104_resize_compress")


def build_task104_gather():
    nodes = [
        constant_node("bank_value", "bank", output_bank()),
        *green_crop_nodes(),
        constant_node("row_idx", "row_idx", np.array([0, 2], dtype=np.int64)),
        constant_node("col_idx", "col_idx", np.array([0, 2], dtype=np.int64)),
        helper.make_node("Gather", ["green_3x3", "row_idx"], ["green_rows"], name="green_rows", axis=2),
        helper.make_node("Gather", ["green_rows", "col_idx"], ["corners_2x2"], name="corners_2x2", axis=3),
        *reshape_bool_compress_tail("corners_2x2"),
    ]
    return make_model(nodes, "task104_gather_compress")


def build_task104_gridsample():
    nodes = [
        constant_node("bank_value", "bank", output_bank()),
        *green_crop_nodes(),
        helper.make_node(
            "GridSample",
            ["green_3x3", "grid"],
            ["corners_2x2"],
            name="sample_corners",
            mode="nearest",
            padding_mode="zeros",
            align_corners=1,
        ),
        *reshape_bool_compress_tail("corners_2x2"),
    ]
    initializers = [
        tensor(
            "grid",
            np.array(
                [[
                    [[-1.0, -1.0], [1.0, -1.0]],
                    [[-1.0, 1.0], [1.0, 1.0]],
                ]],
                dtype=np.float32,
            ),
        )
    ]
    return make_model(nodes, "task104_gridsample_compress", initializers)


def build_task104_topk():
    nodes = [
        constant_node("bank_value", "bank", output_bank()),
        *green_crop_nodes(),
        constant_node("row_idx", "row_idx", np.array([0, 2], dtype=np.int64)),
        constant_node("col_idx", "col_idx", np.array([0, 2], dtype=np.int64)),
        helper.make_node("Gather", ["green_3x3", "row_idx"], ["green_rows"], name="green_rows", axis=2),
        helper.make_node("Gather", ["green_rows", "col_idx"], ["corners_2x2"], name="corners_2x2", axis=3),
        constant_node("reshape4", "reshape4", np.array([4], dtype=np.int64)),
        helper.make_node("Reshape", ["corners_2x2", "reshape4"], ["corners_flat"], name="corners_flat"),
        constant_node("k_value", "k_value", np.array([1], dtype=np.int64)),
        helper.make_node("TopK", ["corners_flat", "k_value"], ["top_value", "quadrant_index"], name="topk", axis=0),
        helper.make_node("Gather", ["bank", "quadrant_index"], ["output"], name="select_bank", axis=0),
    ]
    return make_model(nodes, "task104_topk_gather")


In [ ]:
def build_where_broadcast_bug():
    nodes = [
        constant_node("where_cond", "where_cond", np.array(True, dtype=np.bool_)),
        constant_node("x_starts", "x_starts", np.array([0], dtype=np.int64)),
        constant_node("x_ends", "x_ends", np.array([1], dtype=np.int64)),
        constant_node("x_axes", "x_axes", np.array([2], dtype=np.int64)),
        constant_node("y_starts", "y_starts", np.array([0], dtype=np.int64)),
        constant_node("y_ends", "y_ends", np.array([1], dtype=np.int64)),
        constant_node("y_axes", "y_axes", np.array([3], dtype=np.int64)),
        helper.make_node("Slice", ["input", "x_starts", "x_ends", "x_axes"], ["x"], name="x"),
        helper.make_node("Slice", ["input", "y_starts", "y_ends", "y_axes"], ["y"], name="y"),
        helper.make_node("Where", ["where_cond", "x", "y"], ["output"], name="where"),
    ]
    graph = helper.make_graph(
        nodes,
        "where_broadcast_bug",
        [helper.make_tensor_value_info("input", TensorProto.FLOAT, MODEL_SHAPE)],
        [helper.make_tensor_value_info("output", TensorProto.FLOAT, MODEL_SHAPE)],
    )
    model = helper.make_model(graph, opset_imports=[helper.make_opsetid("", 16)])
    onnx.checker.check_model(model)
    return model


def build_constant_of_shape_bug():
    nodes = [
        constant_node("row_idx", "row_idx", np.array([0], dtype=np.int64)),
        constant_node("col_idx", "col_idx", np.array([0], dtype=np.int64)),
        constant_node("zero_value", "zero_value", np.array(0.0, dtype=np.float32)),
        constant_node("base_shape", "base_shape", np.array([1, 10, 30, 30], dtype=np.int64)),
        helper.make_node("Gather", ["input", "row_idx"], ["row0"], name="row0", axis=2),
        helper.make_node("Gather", ["row0", "col_idx"], ["cell"], name="cell", axis=3),
        helper.make_node("ReduceSum", ["cell"], ["cell_sum"], name="cell_sum", keepdims=0),
        helper.make_node("Greater", ["cell_sum", "zero_value"], ["is_nonzero"], name="is_nonzero"),
        helper.make_node("Cast", ["is_nonzero"], ["scale"], name="scale", to=TensorProto.INT64),
        helper.make_node("Mul", ["scale", "base_shape"], ["runtime_shape"], name="runtime_shape"),
        helper.make_node("ConstantOfShape", ["runtime_shape"], ["output"], name="constant_of_shape"),
    ]
    graph = helper.make_graph(
        nodes,
        "constant_of_shape_bug",
        [helper.make_tensor_value_info("input", TensorProto.FLOAT, MODEL_SHAPE)],
        [helper.make_tensor_value_info("output", TensorProto.FLOAT, MODEL_SHAPE)],
    )
    model = helper.make_model(graph, opset_imports=[helper.make_opsetid("", 16)])
    onnx.checker.check_model(model)
    return model


## Under 9000: Resize + Compress

Task `104`, cost `72`, score `20.723333880984`.

This section is one concrete use of the same broken `Compress` accounting shown again below. The core bug is that, in a correct profile, the final `Compress` should materialize exactly one full board, so the last row should be `out=[1, 10, 30, 30]` and should pay for that output tensor. Instead, `onnx-tool` runs `Compress.value_infer(...)` during `shape_infer(None)` on fake all-zero tensors, so the condition becomes all false and the same node is profiled as `out=[0, 10, 30, 30]` with zero output memory. What makes this specific variant useful is the witness-extraction step before `Compress`: nearest-neighbor `Resize` is also underprofiled here and contributes zero MACs. Relevant code: [graph.py L1023-L1035](https://github.com/ThanatosShinji/onnx-tool/blob/main/onnx_tool/graph.py#L1023-L1035), [tensor.py L433-L438](https://github.com/ThanatosShinji/onnx-tool/blob/main/onnx_tool/tensor.py#L433-L438), [Resize.profile](https://github.com/ThanatosShinji/onnx-tool/blob/main/onnx_tool/node.py#L1220-L1230), [CompressNode](https://github.com/ThanatosShinji/onnx-tool/blob/main/onnx_tool/node.py#L1908-L1915).


In [ ]:
model_path = save_model(build_task104_resize(), "task104_resize.onnx")
report_real_task(104, model_path)
show_expected_vs_actual(
    "The last row should be select_bank with out=[1, 10, 30, 30] and nonzero output memory.",
    "The last row is select_bank with out=[0, 10, 30, 30] and memory=0."
)
show_profile(model_path)
write_submission(model_path, 104, "task104_resize_compress")


## Under 9000: Gather + Compress

Task `104`, cost `96`, score `20.435651808532`.

This is the same core `Compress` bug as the `Resize + Compress` section, but with a different witness-extraction step before the broken selector. In a correct profile, the selector should still end with one full `out=[1, 10, 30, 30]` board. Instead, fake all-zero tensors drive the condition to all false during `shape_infer(None)`, so the last `Compress` row becomes `out=[0, 10, 30, 30]` and pays no output memory at all. What makes this variant separately useful is that `GatherNode` has shape and value inference but no custom `profile(...)`, so the extraction of the witness cells is also charged as zero MACs. Relevant code: [Node.profile](https://github.com/ThanatosShinji/onnx-tool/blob/main/onnx_tool/node.py#L212-L213), [GatherNode](https://github.com/ThanatosShinji/onnx-tool/blob/main/onnx_tool/node.py#L908-L929), [graph.py L1023-L1035](https://github.com/ThanatosShinji/onnx-tool/blob/main/onnx_tool/graph.py#L1023-L1035), [tensor.py L433-L438](https://github.com/ThanatosShinji/onnx-tool/blob/main/onnx_tool/tensor.py#L433-L438), [CompressNode](https://github.com/ThanatosShinji/onnx-tool/blob/main/onnx_tool/node.py#L1908-L1915).


In [ ]:
model_path = save_model(build_task104_gather(), "task104_gather.onnx")
report_real_task(104, model_path)
show_expected_vs_actual(
    "The last row should be select_bank with out=[1, 10, 30, 30] and nonzero output memory.",
    "The last row is select_bank with out=[0, 10, 30, 30] and memory=0."
)
show_profile(model_path)
write_submission(model_path, 104, "task104_gather_compress")


## Under 9000: GridSample + Compress

Task `104`, cost `112`, score `20.281501128705`.

This is again the same core `Compress` bug, but combined with a third witness-extraction path before the broken selector. The `Compress` undercount is unchanged: in a correct profile, the last row should be one full board with `out=[1, 10, 30, 30]`, but `shape_infer(None)` evaluates the condition on synthetic zero tensors and turns the row into `out=[0, 10, 30, 30]` with zero output memory. What makes this variant separately useful is that `GridSampleNode` has shape inference but no custom `profile(...)`, so the corner-sampling step is also charged as zero MACs. Relevant code: [Node.profile](https://github.com/ThanatosShinji/onnx-tool/blob/main/onnx_tool/node.py#L212-L213), [GridSampleNode](https://github.com/ThanatosShinji/onnx-tool/blob/main/onnx_tool/node.py#L1006-L1014), [graph.py L1023-L1035](https://github.com/ThanatosShinji/onnx-tool/blob/main/onnx_tool/graph.py#L1023-L1035), [tensor.py L433-L438](https://github.com/ThanatosShinji/onnx-tool/blob/main/onnx_tool/tensor.py#L433-L438), [CompressNode](https://github.com/ThanatosShinji/onnx-tool/blob/main/onnx_tool/node.py#L1908-L1915).


In [ ]:
model_path = save_model(build_task104_gridsample(), "task104_gridsample.onnx")
report_real_task(104, model_path)
show_expected_vs_actual(
    "The last row should be select_bank with out=[1, 10, 30, 30] and nonzero output memory.",
    "The last row is select_bank with out=[0, 10, 30, 30] and memory=0."
)
show_profile(model_path)
write_submission(model_path, 104, "task104_gridsample_compress")


## Near 9000: TopK

Task `104`, cost `9104`, score `15.883530843634`.

This is a simpler profiling bug: `TopK` is the ranking step that converts the 4-value witness into the chosen board index, but `onnx-tool` assigns it zero MACs because `TopKNode` has no custom `profile(...)` and falls back to the generic base profiler. In a correct profile, the `topk` row should show some nonzero work for ranking. Instead it is profiled as `macs=0`, and almost the entire cost comes only from the final `Gather` materializing the full board. Relevant code: [Node.profile](https://github.com/ThanatosShinji/onnx-tool/blob/main/onnx_tool/node.py#L212-L213), [TopKNode](https://github.com/ThanatosShinji/onnx-tool/blob/main/onnx_tool/node.py#L1856-L1879), [GatherNode](https://github.com/ThanatosShinji/onnx-tool/blob/main/onnx_tool/node.py#L908-L929).


In [ ]:
model_path = save_model(build_task104_topk(), "task104_topk.onnx")
report_real_task(104, model_path)
show_expected_vs_actual(
    "The topk row should contribute nonzero MACs, because ranking values to choose the winning board is real work.",
    "The topk row shows macs=0, so only the final Gather dominates the cost."
)
show_profile(model_path)
write_submission(model_path, 104, "task104_topk")


## The Rest: Where Broadcast Miscount

This is a direct shape-inference bug. ONNX `Where` uses broadcasting, so `x=[1, 10, 1, 30]` and `y=[1, 10, 30, 1]` should produce a full `out=[1, 10, 30, 30]` tensor. `onnx-tool` does not compute broadcast shape here. It calls `_max_shape((cond_shape, x_shape, y_shape))`, which can only return one of the input shapes and therefore cannot ever produce a broadcast result that is larger than every input. That is why the `where` row below is profiled as a thin `out=[1, 10, 1, 30]` slice even though runtime returns the full board. Relevant code: [_max_shape](https://github.com/ThanatosShinji/onnx-tool/blob/main/onnx_tool/node.py#L80-L91), [WhereNode.shape_infer](https://github.com/ThanatosShinji/onnx-tool/blob/main/onnx_tool/node.py#L779-L785).


In [ ]:
model_path = save_model(build_where_broadcast_bug(), "where_broadcast_miscount_demo.onnx")
show_expected_vs_actual(
    "The where row should be out=[1, 10, 30, 30], because [1, 10, 1, 30] and [1, 10, 30, 1] broadcast to the full board.",
    "The where row is out=[1, 10, 1, 30], so only one thin slice is charged."
)
show_demo(model_path)
show_profile(model_path)

## The Rest: ConstantOfShape Fake-Zero Shape

This bug comes from executing value-dependent shape logic on fake data. The final `ConstantOfShape` shape is produced by `runtime_shape`, and `runtime_shape` depends on the input. During `shape_infer(None)`, `onnx-tool` fabricates all-zero tensors for unresolved values and then runs `value_infer(...)` on shape-calc nodes. That makes the guard evaluate to zero, so `runtime_shape` becomes `[0, 0, 0, 0]` and the last row below is profiled as an empty tensor. At runtime, the same graph produces a real `[1, 10, 30, 30]` output, so the profiler is plainly charging the wrong shape. Relevant code: [graph.py L1023-L1035](https://github.com/ThanatosShinji/onnx-tool/blob/main/onnx_tool/graph.py#L1023-L1035), [tensor.py L433-L438](https://github.com/ThanatosShinji/onnx-tool/blob/main/onnx_tool/tensor.py#L433-L438), [ConstantOfShapeNode](https://github.com/ThanatosShinji/onnx-tool/blob/main/onnx_tool/node.py#L2689-L2705).


In [ ]:
model_path = save_model(build_constant_of_shape_bug(), "constantofshape_miscount_demo.onnx")
show_expected_vs_actual(
    "The constant_of_shape row should be out=[1, 10, 30, 30], because runtime_shape becomes that full shape at runtime.",
    "The constant_of_shape row is out=[0, 0, 0, 0], because shape_infer(None) drove the guard with fake zero tensors."
)
show_demo(model_path)
show_profile(model_path)
